# 🚀 02 — Training YOLOv11n
**Tujuan:** Fine-tune YOLOv11n (atau YOLOv8n) pada dataset TACO.

> ⚠️ **Catatan:** Training di CPU akan LAMBAT (beberapa jam untuk 100 epoch).
> Disarankan menggunakan Google Colab dengan GPU T4 untuk training.
> Setelah training selesai, copy `best.pt` ke folder `weights/`.


In [ ]:
import sys
sys.path.insert(0, '..')

import os
from pathlib import Path
import yaml
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'MPS available  : {torch.backends.mps.is_available()}')

# Check ultralytics
try:
    import ultralytics
    print(f'Ultralytics    : {ultralytics.__version__}')
except ImportError:
    print('❌ ultralytics not installed — pip install ultralytics')


## Step 1: Check Dataset

In [ ]:
data_yaml = Path('../data/yolo/data.yaml')
print(f'Dataset config: {data_yaml}')
print(f'Exists        : {data_yaml.exists()}')

if data_yaml.exists():
    with open(data_yaml) as f:
        cfg = yaml.safe_load(f)
    print('\nDataset config:')
    for k, v in cfg.items():
        print(f'  {k}: {v}')

    # Count images
    for split in ['train', 'val', 'test']:
        img_dir = Path(cfg['path']) / 'images' / split
        n = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
        print(f'  {split}: {n} images')


## Step 2: Load Model

In [ ]:
from ultralytics import YOLO

# Try YOLOv11n first, fallback to YOLOv8n
model_candidates = ['yolo11n.pt', 'yolov8n.pt']
model = None
model_name_used = None

for candidate in model_candidates:
    try:
        model = YOLO(candidate)
        model_name_used = candidate
        print(f'✅ Loaded: {candidate}')
        break
    except Exception as e:
        print(f'⚠️  {candidate}: {e}')

if model is None:
    raise RuntimeError('Could not load any YOLO model')

# Model info
print(f'\nModel: {model_name_used}')
print(f'Parameters: {sum(p.numel() for p in model.model.parameters()):,}')


## Step 3: Configure Training

In [ ]:
# Training configuration
TRAIN_CONFIG = {
    'data': str(data_yaml.resolve()),
    'epochs': 100,
    'imgsz': 640,
    'batch': 8,
    'device': 'mps' if torch.backends.mps.is_available() else ('0' if torch.cuda.is_available() else 'cpu'),
    'workers': 2,
    'project': '../runs/train',
    'name': 'taco_yolo11n_v1',
    'patience': 15,
    'lr0': 0.001,
    'lrf': 0.01,
    'optimizer': 'AdamW',
    'cos_lr': True,
    'amp': False,
    'save': True,
    'save_period': 10,
    'plots': True,
    'mosaic': 1.0,
    'mixup': 0.1,
    'copy_paste': 0.1,
    'degrees': 10.0,
    'flipud': 0.1,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.7,
    'hsv_v': 0.4,
}

print('Training configuration:')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')


## Step 4: Fine-tune

In [ ]:
import time

print('🚀 Starting fine-tuning...')
print(f'   Device: {TRAIN_CONFIG["device"]}')
print('   This may take several hours on CPU...')

t_start = time.time()
results = model.train(**TRAIN_CONFIG)
t_elapsed = time.time() - t_start

print(f'\n✅ Training complete in {t_elapsed/3600:.2f} hours')


## Step 5: Save Best Model

In [ ]:
import shutil

# Find best weights
run_dir = Path(f'../runs/train/{TRAIN_CONFIG["name"]}')
best_pt = run_dir / 'weights' / 'best.pt'

if best_pt.exists():
    weights_dir = Path('../weights')
    weights_dir.mkdir(exist_ok=True)
    shutil.copy2(best_pt, weights_dir / 'best.pt')
    print(f'✅ Best model saved: weights/best.pt')
    print(f'   Size: {best_pt.stat().st_size / 1024 / 1024:.1f} MB')
else:
    print('⚠️  best.pt not found. Check run directory.')


## Step 6: Plot Training Curves

In [ ]:
from src.utils.visualization import plot_training_curves

results_csv = run_dir / 'results.csv'
if results_csv.exists():
    plot_training_curves(
        str(results_csv),
        output_dir=Path('../results/visualizations'),
        title=f'Training Progress — {model_name_used} on TACO'
    )
    print('✅ Training curves saved to results/visualizations/')
else:
    print(f'Results CSV not found: {results_csv}')


## Step 7: Quick Validation

In [ ]:
# Quick validation on val set
best_model = YOLO('../weights/best.pt')

# Apply edge constraints for fair evaluation
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
torch.set_num_threads(2)

val_results = best_model.val(
    data=str(data_yaml.resolve()),
    split='val',
    imgsz=640,
    conf=0.25,
    iou=0.5,
    device='cpu',
    plots=True,
    verbose=True
)

print('\n📊 Validation Results:')
print(f'  mAP@0.5     : {val_results.box.map50:.4f}')
print(f'  mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'  Precision   : {val_results.box.mp:.4f}')
print(f'  Recall      : {val_results.box.mr:.4f}')


---

✅ **Training complete!**

Next step: Run `03_inference_evaluation.ipynb` for SAHI inference benchmark and comparison.